# Entrenamiento GNN con Split Random: GCN y LightGCN

**Dataset:** Twitter15 + Twitter16 unificado con **split random estratificado**  
**Objetivo:** Entrenar modelos GNN con split correcto que garantiza overlap de items

En este notebook vamos a entrenar tres modelos:
- GCN con BERT embeddings (3 capas)
- GCN con Random embeddings (3 capas)
- LightGCN (3 capas)

## Cambios principales vs versión anterior:

1. **Split Random Estratificado** (en lugar de temporal)
   - Cada item divide SUS interacciones 80/10/10
   - Todos los items aparecen en train (0% cold-start)
   - 95% de items aparecen en los 3 splits

2. **Entrenamiento Optimizado** con mini-batches
   - 50-100x más rápido que versión anterior
   - Usa `training_utils.py` con DataLoader de PyTorch

3. **Arquitectura de 3 capas** para todos los modelos

## 1. Setup e Instalación

Descargamos los grafos corregidos y configuramos el entorno.

In [ ]:
!mkdir -p graphs_random
!mkdir -p data_processing/processed_round2

# Grafos con split random correcto
!wget -q https://raw.githubusercontent.com/aLotOfGluten/IIC3633-Proyecto/refs/heads/main/midterm/graphs_random/bipartite_graph.pt -P graphs_random/
!wget -q https://raw.githubusercontent.com/aLotOfGluten/IIC3633-Proyecto/refs/heads/main/midterm/graphs_random/social_graph.pt -P graphs_random/

# Interacciones (splits)
!wget -q https://raw.githubusercontent.com/aLotOfGluten/IIC3633-Proyecto/refs/heads/main/midterm/graphs_random/train_interactions.csv -P graphs_random/
!wget -q https://raw.githubusercontent.com/aLotOfGluten/IIC3633-Proyecto/refs/heads/main/midterm/graphs_random/val_interactions.csv -P graphs_random/
!wget -q https://raw.githubusercontent.com/aLotOfGluten/IIC3633-Proyecto/refs/heads/main/midterm/graphs_random/test_interactions.csv -P graphs_random/

# Mapeos
!wget -q https://raw.githubusercontent.com/aLotOfGluten/IIC3633-Proyecto/refs/heads/main/midterm/graphs_random/user_map.csv -P graphs_random/
!wget -q https://raw.githubusercontent.com/aLotOfGluten/IIC3633-Proyecto/refs/heads/main/midterm/graphs_random/item_map.csv -P graphs_random/

# Negative sampling
!wget -q https://raw.githubusercontent.com/aLotOfGluten/IIC3633-Proyecto/refs/heads/main/data_processing/processed_round2/negative_samples.csv -P data_processing/processed_round2/

# Datos procesados (para generar embeddings)
!wget -q https://raw.githubusercontent.com/aLotOfGluten/IIC3633-Proyecto/refs/heads/main/data_processing/processed_round2/twitter15_processed.csv -P data_processing/processed_round2/
!wget -q https://raw.githubusercontent.com/aLotOfGluten/IIC3633-Proyecto/refs/heads/main/data_processing/processed_round2/twitter16_processed.csv -P data_processing/processed_round2/

# Training utilities
!wget -q https://raw.githubusercontent.com/aLotOfGluten/IIC3633-Proyecto/refs/heads/main/midterm/training_utils.py

print("✓ Archivos descargados")

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q torch-geometric
!pip install -q sentence-transformers

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, LGConv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import random
from collections import defaultdict

# Import training utilities
from training_utils import create_train_loader, train_epoch_batched

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

## 2. Carga de Grafos y Datos

Cargamos los grafos con split random correcto. Este split garantiza que:
- Todos los items están en train
- Val y test tienen items que el modelo ya vio (0% cold-start)
- Cada item divide sus interacciones aleatoriamente 80/10/10

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Cargar grafos
bipartite = torch.load('graphs_random/bipartite_graph.pt')
social_graph = torch.load('graphs_random/social_graph.pt')

# Cargar interacciones
train_df = pd.read_csv('graphs_random/train_interactions.csv')
val_df = pd.read_csv('graphs_random/val_interactions.csv')
test_df = pd.read_csv('graphs_random/test_interactions.csv')

# Cargar mapeos
user_map = pd.read_csv('graphs_random/user_map.csv')
item_map = pd.read_csv('graphs_random/item_map.csv')

# Cargar negative samples
negative_samples = pd.read_csv('data_processing/processed_round2/negative_samples.csv')

num_users = bipartite['num_users']
num_items = bipartite['num_items']

print(f"\nDataset stats:")
print(f"  Users: {num_users:,}")
print(f"  Items: {num_items:,}")
print(f"  Train: {len(train_df):,} interactions")
print(f"  Val: {len(val_df):,} interactions")
print(f"  Test: {len(test_df):,} interactions")
print(f"  Negative samples: {len(negative_samples):,}")

# Verificar overlap de items (debe ser ~100%)
train_items = set(train_df['item_idx'].unique())
val_items = set(val_df['item_idx'].unique())
test_items = set(test_df['item_idx'].unique())

print(f"\nItem overlap verification:")
print(f"  Items in train: {len(train_items):,}")
print(f"  Items in val: {len(val_items):,}")
print(f"  Items in test: {len(test_items):,}")
print(f"  Items in train AND val: {len(train_items & val_items):,} ({len(train_items & val_items)/len(val_items)*100:.1f}%)")
print(f"  Items in train AND test: {len(train_items & test_items):,} ({len(train_items & test_items)/len(test_items)*100:.1f}%)")
print(f"  ✓ Zero cold-start: {len(train_items & test_items) == len(test_items)}")

## 3. Construcción del Grafo Bipartito

Construimos el grafo usando SOLO train edges para evitar data leakage.

In [ ]:
print("Construyendo train_edge_index con SOLO train edges...")

train_edges = []
for _, row in train_df.iterrows():
    u, i = row['user_idx'], row['item_idx']
    train_edges.append([u, num_users + i])
    train_edges.append([num_users + i, u])

train_edge_index = torch.LongTensor(train_edges).t().contiguous().to(device)

print(f"train_edge_index shape: {train_edge_index.shape}")
print(f"train_edge_index max node: {train_edge_index.max()}")
print(f"Expected max node: {num_users + num_items - 1}")

## 4. Preparar Negative Sampling

Mapeamos los negative samples a índices numéricos.

In [ ]:
user_id_to_idx = dict(zip(user_map['user_id'], user_map['user_idx']))
item_id_to_idx = dict(zip(item_map['item_id'], item_map['item_idx']))

negative_samples['user_idx'] = negative_samples['user_id'].map(user_id_to_idx)
negative_samples['item_idx'] = negative_samples['item_id'].map(item_id_to_idx)

negative_samples = negative_samples.dropna(subset=['user_idx', 'item_idx'])
negative_samples['user_idx'] = negative_samples['user_idx'].astype(int)
negative_samples['item_idx'] = negative_samples['item_idx'].astype(int)

neg_dict = defaultdict(list)
for _, row in negative_samples.iterrows():
    neg_dict[row['user_idx']].append(row['item_idx'])

print(f"Negative samples mapped: {len(negative_samples):,}")
print(f"Users with negatives: {len(neg_dict):,}")
print(f"Avg negatives per user: {len(negative_samples)/len(neg_dict):.2f}")

## 5. Generación de Embeddings de Items

Creamos dos tipos de embeddings:
1. **BERT embeddings** usando sentence-transformers
2. **Random embeddings** con Xavier initialization

In [ ]:
from sentence_transformers import SentenceTransformer

# Cargar datos para obtener textos
df15 = pd.read_csv('data_processing/processed_round2/twitter15_processed.csv', sep=';')
df16 = pd.read_csv('data_processing/processed_round2/twitter16_processed.csv', sep=';')
df = pd.concat([df15, df16], ignore_index=True)

item_texts = df.groupby('source_tree_id')['text'].first().to_dict()

ordered_texts = []
for item_id in item_map.sort_values('item_idx')['item_id']:
    text = item_texts.get(item_id, "")
    if pd.isna(text) or text == "":
        text = "empty tweet"
    ordered_texts.append(text)

print(f"Generando BERT embeddings para {len(ordered_texts)} items...")
model_bert = SentenceTransformer('all-MiniLM-L6-v2')
item_embeddings_bert = model_bert.encode(ordered_texts, show_progress_bar=True, convert_to_tensor=True)
item_embeddings_bert = item_embeddings_bert.clone().detach().to(device)

print(f"Generando random embeddings...")
item_embeddings_random = torch.empty(num_items, 64)
nn.init.xavier_uniform_(item_embeddings_random)
item_embeddings_random = item_embeddings_random.to(device)

print(f"\nEmbedding shapes:")
print(f"  BERT: {item_embeddings_bert.shape}")
print(f"  Random: {item_embeddings_random.shape}")

## 6. Definición de Modelos

Implementamos GCN y LightGCN con 3 capas cada uno.

In [ ]:
class GCNRecommender(nn.Module):
    def __init__(self, num_users, num_items, item_feature_dim,
                 embedding_dim=128, hidden_dim=64):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim

        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_projection = nn.Linear(item_feature_dim, embedding_dim)

        self.conv1 = GCNConv(embedding_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, embedding_dim)

        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_projection.weight)

    def forward(self, edge_index, item_features):
        user_emb = self.user_embedding.weight
        item_emb = self.item_projection(item_features)
        x = torch.cat([user_emb, item_emb], dim=0)

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.conv3(x, edge_index)

        user_emb_final = x[:self.num_users]
        item_emb_final = x[self.num_users:]

        return user_emb_final, item_emb_final


class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=128, num_layers=3):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.num_layers = num_layers

        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.convs = nn.ModuleList([LGConv() for _ in range(num_layers)])

        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_embedding.weight)

    def forward(self, edge_index):
        x = torch.cat([self.user_embedding.weight, self.item_embedding.weight], dim=0)
        all_embeddings = [x]

        for conv in self.convs:
            x = conv(x, edge_index)
            all_embeddings.append(x)

        final_emb = torch.stack(all_embeddings, dim=0).mean(dim=0)
        user_emb = final_emb[:self.num_users]
        item_emb = final_emb[self.num_users:]

        return user_emb, item_emb

print("Modelos definidos correctamente")

## 7. Funciones de Evaluación

Evaluamos MRR, ILD y Coverage.

In [ ]:
@torch.no_grad()
def evaluate_model(model, edge_index, item_features, test_df, train_df, k=10, device='cpu', is_lightgcn=False):
    model.eval()

    if is_lightgcn:
        user_emb, item_emb = model(edge_index)
    else:
        user_emb, item_emb = model(edge_index, item_features)

    train_items_per_user = train_df.groupby('user_idx')['item_idx'].apply(set).to_dict()

    recommendations = []
    ground_truth = []

    for user_idx in range(model.num_users):
        scores = torch.matmul(user_emb[user_idx], item_emb.t())

        if user_idx in train_items_per_user:
            train_items = list(train_items_per_user[user_idx])
            scores[train_items] = -float('inf')

        _, top_items = torch.topk(scores, k)
        recommendations.append(top_items.cpu().tolist())

        true_items = test_df[test_df['user_idx'] == user_idx]['item_idx'].values
        ground_truth.append(set(true_items))

    return recommendations, ground_truth


def compute_metrics(recommendations, ground_truth, num_items, sample_size=5000):
    reciprocal_ranks = []
    for rec_list, true_items in zip(recommendations, ground_truth):
        rank = None
        for i, item in enumerate(rec_list, 1):
            if item in true_items:
                rank = i
                break
        reciprocal_ranks.append(1.0 / rank if rank else 0.0)
    mrr = np.mean(reciprocal_ranks)

    sampled_indices = np.random.choice(len(recommendations), min(sample_size, len(recommendations)), replace=False)
    sampled_recs = [recommendations[i] for i in sampled_indices]

    similarities = []
    for i in range(len(sampled_recs)):
        for j in range(i + 1, len(sampled_recs)):
            set_i = set(sampled_recs[i])
            set_j = set(sampled_recs[j])
            intersection = len(set_i & set_j)
            union = len(set_i | set_j)
            jaccard = intersection / union if union > 0 else 0.0
            similarities.append(jaccard)
    ild = 1.0 - np.mean(similarities)

    recommended_items = set()
    for rec_list in recommendations:
        recommended_items.update(rec_list)
    coverage = len(recommended_items) / num_items

    return {'MRR': mrr, 'ILD': ild, 'Coverage': coverage}

print("Funciones de evaluación listas")

## 8. Parámetros de Entrenamiento Globales

In [ ]:
EPOCHS = 200
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
EMBED_DIM_GNN = 64
EMBED_DIM_BERT = 128
BATCH_SIZE = 1024  # Para entrenamiento optimizado

print(f"Parámetros de entrenamiento:")
print(f"  Épocas: {EPOCHS}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Weight Decay (L2): {WEIGHT_DECAY}")
print(f"  Embedding Dim (GNN): {EMBED_DIM_GNN}")
print(f"  Embedding Dim (BERT): {EMBED_DIM_BERT}")
print(f"  Batch Size: {BATCH_SIZE}")

## 9. Entrenamiento de GCN-BERT

Primer modelo con embeddings BERT. Usamos mini-batch training optimizado.

In [ ]:
model_gcn_bert = GCNRecommender(
    num_users=num_users,
    num_items=num_items,
    item_feature_dim=item_embeddings_bert.shape[1],
    embedding_dim=EMBED_DIM_BERT,
    hidden_dim=64
).to(device)

optimizer_gcn_bert = torch.optim.Adam(
    model_gcn_bert.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print(f"GCN-BERT (3 capas):")
print(f"  Parámetros: {sum(p.numel() for p in model_gcn_bert.parameters()):,}")

# Crear DataLoader optimizado
train_loader = create_train_loader(train_df, neg_dict, num_items, batch_size=BATCH_SIZE)

losses_gcn_bert = []

print("\nEntrenando GCN-BERT...")
for epoch in tqdm(range(EPOCHS)):
    loss = train_epoch_batched(
        model_gcn_bert,
        train_edge_index,
        item_embeddings_bert,
        train_loader,
        optimizer_gcn_bert,
        device,
        is_lightgcn=False
    )
    losses_gcn_bert.append(loss)

plt.figure(figsize=(10, 4))
plt.plot(losses_gcn_bert)
plt.xlabel('Epoch')
plt.ylabel('BPR Loss')
plt.title(f'Training Loss - GCN-BERT ({EPOCHS} epochs, Batch={BATCH_SIZE})')
plt.grid(True, alpha=0.3)
plt.show()

print("\nEvaluando GCN-BERT...")
recs_gcn_bert, gt_gcn_bert = evaluate_model(
    model_gcn_bert,
    train_edge_index,
    item_embeddings_bert,
    test_df,
    train_df,
    k=10,
    device=device,
    is_lightgcn=False
)
metrics_gcn_bert = compute_metrics(recs_gcn_bert, gt_gcn_bert, num_items)

print("\nMétricas GCN-BERT:")
for k, v in metrics_gcn_bert.items():
    print(f"  {k}: {v:.6f}")

## 10. Entrenamiento de GCN-Random

Segundo modelo con embeddings random.

In [ ]:
model_gcn_random = GCNRecommender(
    num_users=num_users,
    num_items=num_items,
    item_feature_dim=item_embeddings_random.shape[1],
    embedding_dim=EMBED_DIM_GNN,
    hidden_dim=64
).to(device)

optimizer_gcn_random = torch.optim.Adam(
    model_gcn_random.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print(f"GCN-Random (3 capas):")
print(f"  Parámetros: {sum(p.numel() for p in model_gcn_random.parameters()):,}")

# Crear DataLoader
train_loader = create_train_loader(train_df, neg_dict, num_items, batch_size=BATCH_SIZE)

losses_gcn_random = []

print("\nEntrenando GCN-Random...")
for epoch in tqdm(range(EPOCHS)):
    loss = train_epoch_batched(
        model_gcn_random,
        train_edge_index,
        item_embeddings_random,
        train_loader,
        optimizer_gcn_random,
        device,
        is_lightgcn=False
    )
    losses_gcn_random.append(loss)

plt.figure(figsize=(10, 4))
plt.plot(losses_gcn_random)
plt.xlabel('Epoch')
plt.ylabel('BPR Loss')
plt.title(f'Training Loss - GCN-Random ({EPOCHS} epochs, Batch={BATCH_SIZE})')
plt.grid(True, alpha=0.3)
plt.show()

print("\nEvaluando GCN-Random...")
recs_gcn_random, gt_gcn_random = evaluate_model(
    model_gcn_random,
    train_edge_index,
    item_embeddings_random,
    test_df,
    train_df,
    k=10,
    device=device,
    is_lightgcn=False
)
metrics_gcn_random = compute_metrics(recs_gcn_random, gt_gcn_random, num_items)

print("\nMétricas GCN-Random:")
for k, v in metrics_gcn_random.items():
    print(f"  {k}: {v:.6f}")

## 11. Entrenamiento de LightGCN

Tercer modelo: LightGCN (no usa features de items).

In [ ]:
model_lightgcn = LightGCN(
    num_users=num_users,
    num_items=num_items,
    embedding_dim=EMBED_DIM_GNN,
    num_layers=3
).to(device)

optimizer_lightgcn = torch.optim.Adam(
    model_lightgcn.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print(f"LightGCN (3 capas):")
print(f"  Parámetros: {sum(p.numel() for p in model_lightgcn.parameters()):,}")

# Crear DataLoader
train_loader = create_train_loader(train_df, neg_dict, num_items, batch_size=BATCH_SIZE)

losses_lightgcn = []

print("\nEntrenando LightGCN...")
for epoch in tqdm(range(EPOCHS)):
    loss = train_epoch_batched(
        model_lightgcn,
        train_edge_index,
        None,  # LightGCN no usa item_features
        train_loader,
        optimizer_lightgcn,
        device,
        is_lightgcn=True
    )
    losses_lightgcn.append(loss)

plt.figure(figsize=(10, 4))
plt.plot(losses_lightgcn)
plt.xlabel('Epoch')
plt.ylabel('BPR Loss')
plt.title(f'Training Loss - LightGCN ({EPOCHS} epochs, Batch={BATCH_SIZE})')
plt.grid(True, alpha=0.3)
plt.show()

print("\nEvaluando LightGCN...")
recs_lightgcn, gt_lightgcn = evaluate_model(
    model_lightgcn,
    train_edge_index,
    None,
    test_df,
    train_df,
    k=10,
    device=device,
    is_lightgcn=True
)
metrics_lightgcn = compute_metrics(recs_lightgcn, gt_lightgcn, num_items)

print("\nMétricas LightGCN:")
for k, v in metrics_lightgcn.items():
    print(f"  {k}: {v:.6f}")

## 12. Comparación de Modelos

Visualizamos las métricas de los tres modelos.

In [ ]:
comparison_df = pd.DataFrame({
    'GCN-BERT': metrics_gcn_bert,
    'GCN-Random': metrics_gcn_random,
    'LightGCN': metrics_lightgcn
})

print("\n" + "="*70)
print("COMPARACIÓN DE MODELOS (3 capas, split random)")
print("="*70)
print(comparison_df.T.to_string())
print("="*70)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics_names = ['MRR', 'ILD', 'Coverage']
colors = ['#3498db', '#e74c3c', '#2ecc71']

for i, metric in enumerate(metrics_names):
    values = [metrics_gcn_bert[metric], metrics_gcn_random[metric], metrics_lightgcn[metric]]
    axes[i].bar(['BERT', 'Random', 'LightGCN'], values, color=colors)
    axes[i].set_ylabel(metric)
    axes[i].set_title(f'{metric} Comparison')
    axes[i].grid(True, alpha=0.3, axis='y')

    for j, v in enumerate(values):
        axes[i].text(j, v + 0.01, f'{v:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(losses_gcn_bert, label='GCN-BERT', linewidth=2)
plt.plot(losses_gcn_random, label='GCN-Random', linewidth=2)
plt.plot(losses_lightgcn, label='LightGCN', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('BPR Loss')
plt.title('Comparación de Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 13. Análisis de Desinformación

Analizamos qué tipo de contenido recomienda cada modelo.

In [ ]:
labels_df = df.groupby('source_tree_id')['parent_label'].first().reset_index()
labels_df.columns = ['item_id', 'label']
labels_df['item_idx'] = labels_df['item_id'].map(item_id_to_idx)
labels_df = labels_df.dropna(subset=['item_idx'])
labels_df['item_idx'] = labels_df['item_idx'].astype(int)

def analyze_label_distribution(recommendations, labels_df):
    label_counts = {'TR': 0, 'FR': 0, 'UR': 0, 'NR': 0}
    total = 0

    for rec_list in recommendations:
        for item_idx in rec_list:
            item_label = labels_df[labels_df['item_idx'] == item_idx]['label'].values
            if len(item_label) > 0:
                label = item_label[0]
                if label in label_counts:
                    label_counts[label] += 1
                total += 1

    label_percentages = {k: v/total*100 if total > 0 else 0 for k, v in label_counts.items()}
    return label_percentages

dist_gcn_bert = analyze_label_distribution(recs_gcn_bert, labels_df)
dist_gcn_random = analyze_label_distribution(recs_gcn_random, labels_df)
dist_lightgcn = analyze_label_distribution(recs_lightgcn, labels_df)

baseline_dist = labels_df['label'].value_counts(normalize=True).to_dict()
baseline_dist = {k: v*100 for k, v in baseline_dist.items()}

print("\n" + "="*70)
print("DISTRIBUCIÓN DE LABELS EN RECOMENDACIONES")
print("TR=True Rumor, FR=False Rumor, UR=Unverified, NR=Non-Rumor")
print("="*70)

dist_df = pd.DataFrame({
    'Dataset (baseline)': baseline_dist,
    'GCN-BERT': dist_gcn_bert,
    'GCN-Random': dist_gcn_random,
    'LightGCN': dist_lightgcn
})

print(dist_df.T.to_string())
print("="*70)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(4)
width = 0.2

labels_order = ['TR', 'FR', 'UR', 'NR']
baseline_vals = [baseline_dist.get(l, 0) for l in labels_order]
bert_vals = [dist_gcn_bert.get(l, 0) for l in labels_order]
random_vals = [dist_gcn_random.get(l, 0) for l in labels_order]
lightgcn_vals = [dist_lightgcn.get(l, 0) for l in labels_order]

ax.bar(x - 1.5*width, baseline_vals, width, label='Dataset', color='gray', alpha=0.7)
ax.bar(x - 0.5*width, bert_vals, width, label='GCN-BERT', color='#3498db')
ax.bar(x + 0.5*width, random_vals, width, label='GCN-Random', color='#e74c3c')
ax.bar(x + 1.5*width, lightgcn_vals, width, label='LightGCN', color='#2ecc71')

ax.set_ylabel('Porcentaje (%)')
ax.set_title('Distribución de Labels en Recomendaciones Top-10')
ax.set_xticks(x)
ax.set_xticklabels(labels_order)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 14. Guardar Modelos y Resultados

In [ ]:
torch.save(model_gcn_bert.state_dict(), 'model_gcn_bert_random_3layers.pt')
torch.save(model_gcn_random.state_dict(), 'model_gcn_random_random_3layers.pt')
torch.save(model_lightgcn.state_dict(), 'model_lightgcn_random_3layers.pt')

results = {
    'metrics': {
        'gcn_bert': metrics_gcn_bert,
        'gcn_random': metrics_gcn_random,
        'lightgcn': metrics_lightgcn
    },
    'label_distribution': {
        'baseline': baseline_dist,
        'gcn_bert': dist_gcn_bert,
        'gcn_random': dist_gcn_random,
        'lightgcn': dist_lightgcn
    }
}

import json
with open('results_random_split_3layers.json', 'w') as f:
    json.dump(results, f, indent=2)

print("✓ Modelos y resultados guardados")